# meeting_extracted_w2_t4_reconcile — W2 T4：Sales Brief 對照判定（L3）

- 用途：一期一跑。把 T3 標的會議條目（`sb_relevant`）與 Sales Brief 填報文字對成格子 app × brand × facet × brief_month，單邊為空程式判 `meeting_only` / `sb_only`，兩邊都有才呼叫 LLM 判 `corroborate` / `supplement` / `conflict`；每格產 PM 內部摘要與可外送摘要；app 層（brand = ALL）格再彙整跨客戶訊號（規格 §4.4、§5.3、§7.3）。
- 輸入 table：`{catalog}.{schema}.v_meeting_items_sb_q`（004；T3 標註 ⋈ 中立層）、`v_meeting_sb_brief_text`（004；填報端一格一列）、`v_meeting_sb_period`（004；品牌清單）
- 輸出 table：`{catalog}.{schema}.g_meeting_sb_reconcile_{unit}`（004）。MERGE on `(app_name, brand, facet, brief_month)`；**不碰** `reviewer / review_verdict / review_relation / review_note / review_carried_from`，`created_at` 只在首次寫。
- 參數（widgets）：`catalog`、`schema`、`unit`、`brief_month`（空 = 本月）、`prompt_version`（= [c04] `PROMPT_VERSION_IN_CELL`）、`model`、`max_items_per_cell`（8）、`max_items_per_tag_other`（8）、`max_cells`（LLM 費用閘門；0 = 不限）、`summary_batch`（摘要模式一次幾格）、`concurrency`、`app_level`、`prompt_cache`、`dry_run`
- 排程：每月一次，T3 之後、brief 填完後跑。brief 沒填時所有格都是 `meeting_only`（只產摘要，不判 relation）。
- 負責人 / 更新日期：（填）/ 2026-10-23

Cell 標籤規則見 `docs/conventions.md`。

## 格子怎麼來、怎麼判（決定 D35 / D36）

| 情況 | relation | 誰判 | 摘要 |
|---|---|---|---|
| 會議有、填報空（或該期無此品牌、brand = `UNKNOWN`） | `meeting_only` | 程式（`relation_basis = rule`） | 摘要模式批次產 |
| 填報有、會議無 | `sb_only` | 程式 | 無 |
| 兩邊都有 | `corroborate` / `supplement` / `conflict` | LLM 格子模式（`llm`） | 同一次呼叫 |
| ALL 格（app 層） | 同上判準；填報端 = app_summary，會議端 = 各品牌格摘要 | LLM app 層模式；填報空 → `meeting_only` | 同一次呼叫，另產 `signals`（≥ 2 品牌共享）→ `cross_customer_note`，參與品牌格標 `cross_customer_flag` |

每格送 LLM 的條目：依 `confidence` 降冪最多 `max_items_per_cell`；`other` 格改成每個主標籤最多 `max_items_per_tag_other`（supply_chain / quality / allocation / capacity 各一組），其餘只計數。S2 條目在 prompt 標〔S2〕；`meeting_summary_shareable` 若含任何 S2 條目的 `numbers.value` → 置 NULL、`shareable_failed = true`。


In [ ]:
# [c01] params
# W2 T4：一次對照一期。由 job parameters 覆蓋；widget 建立過後改預設值不會生效，要重設先 dbutils.widgets.removeAll()。
import re as _re
from datetime import date as _date

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("unit", "bu")                       # g_meeting_sb_reconcile_{unit}
dbutils.widgets.text("brief_month", "")                  # yyyy-MM；空 = 本月
dbutils.widgets.text("prompt_version", "0.1")            # 要等於 [c04] PROMPT_VERSION_IN_CELL；改 prompt 就升
dbutils.widgets.text("model", "claude-sonnet-5")         # Azure Foundry 上的模型名（同 task 3 / Stage A）
dbutils.widgets.text("max_items_per_cell", "8")          # 規格 §4.4：每格最多帶 8 條（依 confidence 降冪）
dbutils.widgets.text("max_items_per_tag_other", "8")     # other 格改成每個主標籤最多幾條（D36）
dbutils.widgets.text("max_cells", "12")                  # 費用閘門：本次最多呼叫幾次 LLM（格子 ＋ 摘要批 ＋ app 層都算）；0 = 不限
dbutils.widgets.text("summary_batch", "6")               # 摘要模式一次帶幾格
dbutils.widgets.text("concurrency", "4")
dbutils.widgets.dropdown("app_level", "true", ["true", "false"])     # 跑 ALL 格（app 層模式）
dbutils.widgets.dropdown("prompt_cache", "true", ["true", "false"])
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])      # true：照樣呼叫 LLM（受 max_cells 限制）、印關卡與樣本，不寫表

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "unit": dbutils.widgets.get("unit").strip(),
    "brief_month": dbutils.widgets.get("brief_month").strip() or _date.today().strftime("%Y-%m"),
    "prompt_version": dbutils.widgets.get("prompt_version").strip(),
    "model": dbutils.widgets.get("model").strip(),
    "max_items_per_cell": int(dbutils.widgets.get("max_items_per_cell")),
    "max_items_per_tag_other": int(dbutils.widgets.get("max_items_per_tag_other")),
    "max_cells": int(dbutils.widgets.get("max_cells")),
    "summary_batch": int(dbutils.widgets.get("summary_batch")),
    "concurrency": max(1, int(dbutils.widgets.get("concurrency"))),
    "app_level": dbutils.widgets.get("app_level") == "true",
    "prompt_cache": dbutils.widgets.get("prompt_cache") == "true",
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["unit"], "catalog / schema / unit 不可為空"
assert _re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", cfg["brief_month"]), "brief_month 格式 yyyy-MM"
assert cfg["prompt_version"] and cfg["model"], "prompt_version / model 不可為空"
assert 1 <= cfg["max_items_per_cell"] <= 20 and 1 <= cfg["max_items_per_tag_other"] <= 20, "每格條目上限 1–20"
assert cfg["max_cells"] >= 0 and 1 <= cfg["summary_batch"] <= 12 and cfg["concurrency"] <= 16, "max_cells ≥ 0；summary_batch 1–12；concurrency ≤ 16"
print(cfg)


In [ ]:
# [c02] imports
# [c02]～[c06] 是純 Python / 純 DataFrame 函式（不碰 spark session / dbutils / 網路），tests/ 直接載入；[c07] 是 LLM 接線。
# 常數對齊 004 [c02]。
import json
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

APPS = ("TV", "NB", "MNT")
FACETS = ("market_outlook", "product_tech", "price", "other")
FACET_LABEL = {"market_outlook": "市場展望", "product_tech": "產品技術", "price": "價格", "other": "其他"}
BRAND_ALL = "ALL"            # app 層格
BRAND_UNKNOWN = "UNKNOWN"    # 會議端 brand NULL（客戶 NULL）但 app 已解的相關條目：D36，不丟，當一個沒有填報端的品牌格
RELATIONS_LLM = ("corroborate", "supplement", "conflict")
RELATIONS_RULE = ("meeting_only", "sb_only")
RELATIONS = RELATIONS_LLM + RELATIONS_RULE
RULE_MODEL = "rule"
SENS_ORDER = {"S0": 0, "S1": 1, "S2": 2}
MAX_SUMMARY_CHARS = 600
MAX_RATIONALE_CHARS = 300
MAX_SIGNAL_CHARS = 300
MIN_NUMBER_LEN = 2           # shareable 驗證：S2 條目 numbers.value 至少這麼長才比（單一數字 1、2 太常見）
EMAIL_ANY_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)+")
PRICE_PER_MTOK = {"input": 2.0, "output": 10.0, "cache_write": 2.5, "cache_read": 0.2}  # §10.1 只印 log

# 從 v_meeting_items_sb_q 讀的欄（只取 sb_relevant、本期、active）
ITEM_COLS = ["item_id", "item_key", "app_name", "brand", "facet", "brief_month", "window_start", "window_end",
             "topic_tag_primary", "summary", "quote", "event_date", "speaker_side", "customer", "numbers", "model_nos",
             "sensitivity", "confidence", "statement_type"]
# 每格送 LLM 的條目欄（struct 欄位順序 = prompt 顯示順序）
CELL_ITEM_FIELDS = ["item_id", "item_key", "topic_tag_primary", "summary", "quote", "event_date", "speaker_side",
                    "numbers", "model_nos", "sensitivity", "confidence", "statement_type"]
# 對照格表欄（004 [c04] g_meeting_sb_reconcile_bu 順序）；T4 寫的 = 全部扣掉 PROTECTED_COLS
RECONCILE_COLS = ["app_name", "brand", "facet", "brief_month", "window_start", "window_end", "sb_text", "sb_source",
                  "meeting_summary", "meeting_summary_shareable", "shareable_failed", "meeting_item_ids",
                  "meeting_item_keys", "n_items", "n_items_sent", "topic_tags_agg", "relation", "relation_basis",
                  "rationale", "number_caveat", "cross_customer_flag", "cross_customer_note", "sensitivity_max",
                  "reviewer", "review_verdict", "review_relation", "review_note", "review_carried_from",
                  "prompt_version", "model", "created_at", "updated_at"]
PROTECTED_COLS = ("reviewer", "review_verdict", "review_relation", "review_note", "review_carried_from")
WRITE_COLS = [c for c in RECONCILE_COLS if c not in PROTECTED_COLS]
ROW_SCHEMA = T.StructType([
    T.StructField("app_name", T.StringType(), False),
    T.StructField("brand", T.StringType(), False),
    T.StructField("facet", T.StringType(), False),
    T.StructField("brief_month", T.StringType(), False),
    T.StructField("window_start", T.DateType(), True),
    T.StructField("window_end", T.DateType(), True),
    T.StructField("sb_text", T.StringType(), True),
    T.StructField("sb_source", T.StringType(), True),
    T.StructField("meeting_summary", T.StringType(), True),
    T.StructField("meeting_summary_shareable", T.StringType(), True),
    T.StructField("shareable_failed", T.BooleanType(), True),
    T.StructField("meeting_item_ids", T.ArrayType(T.StringType()), True),
    T.StructField("meeting_item_keys", T.ArrayType(T.StringType()), True),
    T.StructField("n_items", T.IntegerType(), True),
    T.StructField("n_items_sent", T.IntegerType(), True),
    T.StructField("topic_tags_agg", T.ArrayType(T.StringType()), True),
    T.StructField("relation", T.StringType(), True),
    T.StructField("relation_basis", T.StringType(), True),
    T.StructField("rationale", T.StringType(), True),
    T.StructField("number_caveat", T.StringType(), True),
    T.StructField("cross_customer_flag", T.BooleanType(), True),
    T.StructField("cross_customer_note", T.StringType(), True),
    T.StructField("sensitivity_max", T.StringType(), True),
    T.StructField("prompt_version", T.StringType(), True),
    T.StructField("model", T.StringType(), True),
])


In [ ]:
# [c03] pure_shared
# 純 Python：字串正規化、LLM JSON 解析（同 Stage A [c05]）、敏感度、條目挑選（每格上限 / other 格每標籤上限）、shareable 驗證。
def norm_space(s: str | None) -> str:
    return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()


def parse_llm_json(text: str | None) -> tuple[dict | None, str | None]:
    """回覆文字 → (dict, None) 或 (None, 錯誤訊息)。容忍 markdown 圍欄與前後說明文字（取第一個 { 到最後一個 }）。"""
    if not text or not text.strip():
        return None, "empty response"
    s = text.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    i, j = s.find("{"), s.rfind("}")
    if i < 0 or j <= i:
        return None, "no JSON object found"
    try:
        obj = json.loads(s[i:j + 1])
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e.msg} at pos {e.pos}"
    if not isinstance(obj, dict):
        return None, "top-level must be an object"
    return obj, None


def str_or_none(v, cap: int) -> str | None:
    """LLM 欄位 → 去空白、截長、email 樣式換 <email>（S3 硬規則）；list / dict / 空 → None。"""
    if v is None or isinstance(v, (list, dict, bool)):
        return None
    s = norm_space(str(v))
    s = EMAIL_ANY_RE.sub("<email>", s)
    return s[:cap] if s else None


def sens_max(levels) -> str | None:
    """['S1', 'S2', None] → 'S2'；全空 → None。"""
    vals = [x for x in levels if x in SENS_ORDER]
    return max(vals, key=lambda x: SENS_ORDER[x]) if vals else None


def number_values(numbers_json: str | None) -> list[str]:
    """中立層 numbers（JSON 字串 [{value, unit, what}]）→ value 字串清單（去空白、長度 ≥ MIN_NUMBER_LEN）。壞 JSON → []。"""
    if not numbers_json:
        return []
    try:
        arr = json.loads(numbers_json)
    except (json.JSONDecodeError, TypeError):
        return []
    out = []
    for x in arr if isinstance(arr, list) else []:
        v = x.get("value") if isinstance(x, dict) else x
        if v is None or isinstance(v, (list, dict, bool)):
            continue
        s = norm_space(str(v)).replace(",", "")
        if len(s) >= MIN_NUMBER_LEN:
            out.append(s)
    return out


def pick_items(items: list[dict], facet: str, k_cell: int, k_tag: int) -> list[dict]:
    """送 LLM 的條目：依 confidence 降冪（同分 item_id 升冪）。other 格每個 topic_tag_primary 最多 k_tag 條（各標籤輪流，
    確保 supply_chain / quality / allocation / capacity 都有代表）；其餘面向整格最多 k_cell 條。"""
    ordered = sorted(items, key=lambda x: (-(x.get("confidence") or 0.0), x.get("item_id") or ""))
    if facet != "other":
        return ordered[:k_cell]
    per_tag: dict[str, list[dict]] = {}
    for it in ordered:
        per_tag.setdefault(it.get("topic_tag_primary") or "", []).append(it)
    out: list[dict] = []
    for tag in sorted(per_tag):
        out.extend(per_tag[tag][:k_tag])
    return sorted(out, key=lambda x: (-(x.get("confidence") or 0.0), x.get("item_id") or ""))


def validate_shareable(shareable: str | None, items: list[dict]) -> tuple[str | None, bool]:
    """§4.4：shareable 內出現任何 S2 條目的 numbers.value → (None, True)。shareable 空 → (None, False)。"""
    s = norm_space(shareable).replace(",", "") if shareable else ""
    if not s:
        return None, False
    for it in items:
        if it.get("sensitivity") != "S2":
            continue
        for v in number_values(it.get("numbers")):
            if v in s:
                return None, True
    return s, False


def cell_key(app: str, brand: str, facet: str) -> str:
    return f"{app}|{brand}|{facet}"


def merge_stats(total: dict, part: dict) -> dict:
    for k, v in part.items():
        if isinstance(v, list):
            total[k] = total.get(k, []) + v
        else:
            total[k] = total.get(k, 0) + v
    return total


In [ ]:
# [c04] prompt_t4
# L3 對照判定 prompt（規格 §5.3 三種模式、§7.1 面向定義、§7.3 relation 判準）。一個 system prompt 共用（prompt caching），
# user prompt 開頭標明模式。條目已 token 化（<Pnn>），prompt 不帶姓名 / email。改這個 cell 就升 [c01] prompt_version。
PROMPT_VERSION_IN_CELL = "0.1"

FACET_DEFS = """【四面向定義（Sales Brief 填報的四格）】
market_outlook 市場展望：需求、客戶策略與態度、商務關係（合資、轉單、合作）
product_tech   產品技術：產品規劃、機種方向、規格與技術選擇
price          價格：報價、售價、成本、價差
other          其他：供應鏈、料況、產能、分配（allocation）、品質要求"""

RELATION_RULES = """【relation 判準（兩邊都有內容時）】
corroborate  兩邊方向一致，會議提供細節或時程（填報說轉向 AG25，會議說 BOM 已升、10 月放量）
supplement   填報該格有內容但未提此事，會議有可行動訊息（填報談存儲缺料，會議有 VLRR 未達客戶要求）
conflict     方向相反，或口徑一致的數字 / 時程不同。**兩邊數字單位或口徑不同時不得以數字差異判 conflict**，須在 number_caveat 說明
只能三選一；沒把握時選 supplement 並在 rationale 說明。"""

SHAREABLE_RULES = """【兩種摘要】
meeting_summary           給 PM 內部：1–3 句綜合會議端在說什麼，可含數字、量價、限簽、Allocation 細節。
meeting_summary_shareable 可外送（可能進 Sales Brief 信件）：**不得引用標記〔S2〕的條目中的任何數字、數量、價格、限簽與 Allocation 細節**，
                          只能以方向描述（例「客戶對 VLRR 提出更嚴的允收要求」而不是「要求 400 dppm」）。沒有 S2 條目時可與 meeting_summary 相同。
兩種摘要都用繁體中文、型號原文、人以 <Pnn> token 指涉，不要嘗試還原任何人的姓名。只根據給你的條目寫，不要補充條目沒有的事。"""

OUTPUT_SCHEMAS = """【輸出】只輸出一個 JSON 物件，不要 markdown 圍欄、不要說明文字。依模式：
mode=cell（一格：填報原文 ＋ 會議條目）
{"relation": "corroborate|supplement|conflict", "meeting_summary": "", "meeting_summary_shareable": "", "rationale": "一句理由", "evidence_item_ids": ["item_id", ...], "number_caveat": "兩邊數字可否比較；無數字寫 null"}
mode=summary（多格：只有會議端，不判 relation）
{"cells": [{"key": "格子 key 照輸入", "meeting_summary": "", "meeting_summary_shareable": ""}]}
每個輸入格子都要有一個對應的 key，不可漏、不可多。
mode=app（app 層：各品牌格的會議摘要 ＋ 填報的整體摘要）
{"relation": "corroborate|supplement|conflict|meeting_only", "meeting_summary": "", "meeting_summary_shareable": "", "signals": [{"note": "跨客戶共同訊號一句", "brands": ["品牌", "品牌"], "item_ids": []}], "rationale": ""}
signals 只收 ≥ 2 個品牌共享的訊號（同一型號、同一供應限制、同一價格方向…）；沒有就給空陣列。填報整體摘要為空時 relation 填 meeting_only。"""

FEW_SHOT = """【示例】（取自規格樣本，只示範判法）
mode=cell｜TV × TCL × product_tech｜填報「轉向 POL AG25/AG25LR，55 吋轉 Trigate」｜會議「LT0 BOM 已升改用 SMK AG25，目標 10 月放量」
→ {"relation": "corroborate", "meeting_summary": "T850 LT0 BOM 已升版改用 SMK AG25 偏光板，目標 10 月放量。", "meeting_summary_shareable": "T850 LT0 已改用 AG25 偏光板，目標 10 月放量。", "rationale": "填報說轉向 AG25，會議給了機種與時程", "evidence_item_ids": ["BU-MTG-..."], "number_caveat": null}
mode=cell｜TV × TCL × other｜填報「存儲缺料導致中小尺寸成本增加」｜會議「已請採購備 AG25 原材 10K」「TCL 要求 VLRR 降到 400 dppm 以下（目前 603）」〔S2〕
→ {"relation": "supplement", "meeting_summary": "已請採購備 AG25 原材 10K；TCL 要求 VLRR 降到 400 dppm 以下（目前 603）。", "meeting_summary_shareable": "已請採購備 AG25 原材；TCL 對 VLRR 提出更嚴的允收要求。", "rationale": "填報談存儲缺料，會議有備料與品質要求兩件可行動訊息", "evidence_item_ids": ["..."], "number_caveat": "填報無數字，會議的 dppm 為品質指標，不構成比較"}
mode=app｜MNT × product_tech｜Acer 格摘要「M270QAN06.A 4ch LVDS UHD 100Hz，MP Jan'27」｜ASUS 填報「M270QAN06.A 4ch LVDS 開案 pending 等價差」
→ {"relation": "supplement", "meeting_summary": "acer 與 ASUS 同時關注 M270QAN06.A 的 4ch LVDS 版本，acer 端已有 MP 時程。", "meeting_summary_shareable": "同左", "signals": [{"note": "acer 與 ASUS 同時關注 M270QAN06.A 4ch LVDS 版本", "brands": ["Acer", "Asus"], "item_ids": ["..."]}], "rationale": "整體摘要未提此型號的跨客戶需求"}"""

SYSTEM_PROMPT_T4 = f"""你是 AUO DSBG「會議紀錄 × Sales Brief 對照」的執行者。Sales Brief 是業務每月填報的四面向摘要（按 app × 品牌）；會議條目是從客戶會議紀錄萃出的原子事實（已 token 化）。你的工作：判斷會議端與填報端的關係（relation），並把會議端寫成摘要。你只產候選，不定案（PM 會審）。

{FACET_DEFS}

{RELATION_RULES}

{SHAREABLE_RULES}

【其他規則】
- evidence_item_ids 只能用輸入裡的 item_id，列出支持 relation 判斷的條目（最多 5 個）。
- rationale 一句話，寫判斷依據，不寫給 PM 的話。
- 條目若互相矛盾，以日期較新者為主，並在 rationale 註明。

{OUTPUT_SCHEMAS}

{FEW_SHOT}"""


def _item_line(it: dict) -> str:
    flag = "〔S2〕" if it.get("sensitivity") == "S2" else ""
    side = it.get("speaker_side") or "unknown"
    parts = [f"- [{it.get('item_id')}]{flag} {it.get('event_date')} {side} | {it.get('statement_type')}/{it.get('topic_tag_primary')}",
             f"  summary: {norm_space(it.get('summary'))}", f"  quote: {norm_space(it.get('quote'))}"]
    nums = norm_space(it.get("numbers")) if it.get("numbers") not in (None, "[]") else ""
    if nums:
        parts.append(f"  numbers: {nums}")
    if it.get("model_nos"):
        parts.append(f"  型號: {', '.join(it['model_nos'])}")
    return "\n".join(parts)


def _cell_head(app: str, brand: str, facet: str, brief_month: str) -> str:
    return f"app {app} × 品牌 {brand} × 面向 {facet}（{FACET_LABEL.get(facet, facet)}）× 期別 {brief_month}"


def build_cell_prompt(cell: dict, items: list[dict]) -> str:
    """mode=cell：一格（兩邊都有）。cell 需有 app_name / brand / facet / brief_month / sb_text。"""
    return (f"mode=cell\n{_cell_head(cell['app_name'], cell['brand'], cell['facet'], cell['brief_month'])}\n\n"
            f"## Sales Brief 填報原文\n{norm_space(cell.get('sb_text'))}\n\n"
            f"## 會議條目（{len(items)} 條）\n" + "\n".join(_item_line(it) for it in items))


def build_summary_prompt(cells: list[tuple[dict, list[dict]]]) -> str:
    """mode=summary：多格（只有會議端），每格帶 key 與條目。"""
    blocks = []
    for cell, items in cells:
        key = cell_key(cell["app_name"], cell["brand"], cell["facet"])
        blocks.append(f"### key={key}\n{_cell_head(cell['app_name'], cell['brand'], cell['facet'], cell['brief_month'])}\n"
                      + "\n".join(_item_line(it) for it in items))
    return f"mode=summary\n共 {len(cells)} 格，每格寫兩種摘要\n\n" + "\n\n".join(blocks)


def build_app_prompt(app: str, facet: str, brief_month: str, brand_cells: list[dict], sb_text: str | None) -> str:
    """mode=app：各品牌格的 meeting_summary（＋證據 id）＋ ALL 格填報原文。brand_cells: [{brand, meeting_summary, item_ids}]。"""
    lines = [f"mode=app\napp {app} × 面向 {facet}（{FACET_LABEL.get(facet, facet)}）× 期別 {brief_month}\n",
             f"## Sales Brief 整體摘要（第 3 節）\n{norm_space(sb_text) if sb_text else '（空白）'}\n",
             f"## 各品牌格的會議摘要（{len(brand_cells)} 格）"]
    for bc in brand_cells:
        ids = ", ".join(bc.get("item_ids") or [])
        lines.append(f"- 品牌 {bc['brand']}：{norm_space(bc.get('meeting_summary'))}" + (f"（item_ids: {ids}）" if ids else ""))
    return "\n".join(lines)


def retry_user_prompt(error: str) -> str:
    return f"上一輪輸出無法使用：{error}。請重新只輸出一個合法的 JSON 物件（同樣的 schema、同樣的模式），不要任何說明文字或 markdown 圍欄。"


In [ ]:
# [c05] pure_postprocess
# LLM 回覆 → 對照格列（ROW_SCHEMA 欄的 dict）。程式說了算：relation 只能三選一（不合法 = 這格作廢、下次再判）；evidence 只留送過的 id；
# shareable 驗證；app 層 signals 只留 ≥ 2 品牌；填報空的 ALL 格 relation 強制 meeting_only。
def base_row(cell: dict, items_sent: list[dict], prompt_version: str, model: str) -> dict:
    """格子 → 列的共同欄（relation / 摘要 / rationale 之後填）。cell 來自 [c06] 的 grid 列（dict）。"""
    all_items = cell.get("items") or []
    return {
        "app_name": cell["app_name"], "brand": cell["brand"], "facet": cell["facet"], "brief_month": cell["brief_month"],
        "window_start": cell.get("window_start"), "window_end": cell.get("window_end"),
        "sb_text": cell.get("sb_text"), "sb_source": cell.get("sb_source") or "none",
        "meeting_summary": None, "meeting_summary_shareable": None, "shareable_failed": False,
        "meeting_item_ids": [it["item_id"] for it in all_items] or None,
        "meeting_item_keys": [it["item_key"] for it in all_items] or None,
        "n_items": len(all_items), "n_items_sent": len(items_sent),
        "topic_tags_agg": cell.get("topic_tags_agg") or None,
        "relation": None, "relation_basis": None, "rationale": None, "number_caveat": None,
        "cross_customer_flag": False, "cross_customer_note": None,
        "sensitivity_max": cell.get("sensitivity_max"), "prompt_version": prompt_version, "model": model,
    }


def rule_row(cell: dict, relation: str, prompt_version: str) -> dict:
    """單邊格：meeting_only（摘要之後由摘要模式補）/ sb_only。"""
    assert relation in RELATIONS_RULE
    row = base_row(cell, [], prompt_version, RULE_MODEL)
    row.update({"relation": relation, "relation_basis": "rule"})
    return row


def _fill_summaries(row: dict, obj: dict, items_sent: list[dict]) -> None:
    row["meeting_summary"] = str_or_none(obj.get("meeting_summary"), MAX_SUMMARY_CHARS)
    shareable, failed = validate_shareable(str_or_none(obj.get("meeting_summary_shareable"), MAX_SUMMARY_CHARS), items_sent)
    row["meeting_summary_shareable"], row["shareable_failed"] = shareable, failed


def postprocess_cell(obj: dict | None, error: str | None, cell: dict, items_sent: list[dict], prompt_version: str,
                     model: str) -> tuple[dict | None, dict]:
    """mode=cell 的回覆 → (列, 統計)。None = 這格作廢（端點失敗 / relation 不合法）。"""
    st: dict = {"cell_sent": 1}
    if obj is None:
        st["cell_failed"] = 1
        st["errors"] = [f"{cell_key(cell['app_name'], cell['brand'], cell['facet'])}: {error}"]
        return None, st
    rel = obj.get("relation")
    rel = rel.strip().lower() if isinstance(rel, str) else ""
    if rel not in RELATIONS_LLM:
        st["cell_failed"] = 1
        st["bad_relation"] = 1
        st["errors"] = [f"{cell_key(cell['app_name'], cell['brand'], cell['facet'])}: bad relation {rel!r}"]
        return None, st
    row = base_row(cell, items_sent, prompt_version, model)
    row.update({"relation": rel, "relation_basis": "llm",
                "rationale": str_or_none(obj.get("rationale"), MAX_RATIONALE_CHARS),
                "number_caveat": str_or_none(obj.get("number_caveat"), MAX_RATIONALE_CHARS)})
    _fill_summaries(row, obj, items_sent)
    sent_ids = {it["item_id"] for it in items_sent}
    ev = obj.get("evidence_item_ids")
    ev = [x for x in ev if isinstance(x, str) and x in sent_ids] if isinstance(ev, list) else []
    if ev:  # 證據排前面，其餘照原順序
        row["meeting_item_ids"] = ev + [x for x in (row["meeting_item_ids"] or []) if x not in ev]
    if row["shareable_failed"]:
        st["shareable_failed"] = 1
    if not row["meeting_summary"]:
        st["summary_missing"] = 1
    st[rel] = 1
    return row, st


def postprocess_summary(obj: dict | None, error: str | None, batch: list[tuple[dict, list[dict]]]) -> tuple[dict, dict]:
    """mode=summary 的回覆 → {key: (meeting_summary, shareable, failed)}；漏掉的格不在結果裡（列仍會寫，摘要 NULL）。"""
    st: dict = {"summary_cells_sent": len(batch)}
    if obj is None:
        st["summary_batch_failed"] = 1
        st["errors"] = [f"summary batch: {error}"]
        return {}, st
    by_key: dict[str, dict] = {}
    for c in obj.get("cells") or []:
        if isinstance(c, dict) and isinstance(c.get("key"), str):
            by_key.setdefault(c["key"], c)
    out: dict[str, tuple] = {}
    for cell, items in batch:
        key = cell_key(cell["app_name"], cell["brand"], cell["facet"])
        c = by_key.get(key)
        if c is None:
            st["summary_missing"] = st.get("summary_missing", 0) + 1
            continue
        summ = str_or_none(c.get("meeting_summary"), MAX_SUMMARY_CHARS)
        shareable, failed = validate_shareable(str_or_none(c.get("meeting_summary_shareable"), MAX_SUMMARY_CHARS), items)
        if failed:
            st["shareable_failed"] = st.get("shareable_failed", 0) + 1
        out[key] = (summ, shareable, failed)
    st["summary_filled"] = len(out)
    return out, st


def postprocess_app(obj: dict | None, error: str | None, all_cell: dict, brand_cells: list[dict], prompt_version: str,
                    model: str) -> tuple[dict | None, set, dict]:
    """mode=app 的回覆 → (ALL 格列, 參與訊號的品牌集合, 統計)。all_cell: grid 的 ALL 列（可能只有填報端或兩邊都空的合成列）。
    填報空 → relation 強制 meeting_only / rule；否則 LLM 三選一（不合法 → 作廢）。"""
    st: dict = {"app_sent": 1}
    key = cell_key(all_cell["app_name"], BRAND_ALL, all_cell["facet"])
    if obj is None:
        st["app_failed"] = 1
        st["errors"] = [f"{key}: {error}"]
        return None, set(), st
    row = base_row(all_cell, [], prompt_version, model)
    ids = [i for bc in brand_cells for i in (bc.get("item_ids") or [])]
    keys = [k for bc in brand_cells for k in (bc.get("item_keys") or [])]
    row["meeting_item_ids"], row["meeting_item_keys"] = (ids or None), (keys or None)
    row["n_items"], row["n_items_sent"] = len(ids), len(brand_cells)
    row["topic_tags_agg"] = sorted({t for bc in brand_cells for t in (bc.get("topic_tags_agg") or [])}) or None
    row["sensitivity_max"] = sens_max([bc.get("sensitivity_max") for bc in brand_cells])
    if not all_cell.get("sb_text"):
        row.update({"relation": "meeting_only", "relation_basis": "rule"})
    else:
        rel = obj.get("relation")
        rel = rel.strip().lower() if isinstance(rel, str) else ""
        if rel not in RELATIONS_LLM:
            st["app_failed"] = 1
            st["bad_relation"] = 1
            st["errors"] = [f"{key}: bad relation {rel!r}"]
            return None, set(), st
        row.update({"relation": rel, "relation_basis": "llm"})
    row["rationale"] = str_or_none(obj.get("rationale"), MAX_RATIONALE_CHARS)
    # shareable 驗證用所有品牌格的 S2 條目
    s2_items = [it for bc in brand_cells for it in (bc.get("items") or []) if it.get("sensitivity") == "S2"]
    _fill_summaries(row, obj, s2_items)
    known = {bc["brand"] for bc in brand_cells}
    notes, flagged = [], set()
    for sg in obj.get("signals") or []:
        if not isinstance(sg, dict):
            continue
        brands = [b for b in (sg.get("brands") or []) if isinstance(b, str) and b in known] if isinstance(sg.get("brands"), list) else []
        note = str_or_none(sg.get("note"), MAX_SIGNAL_CHARS)
        if len(set(brands)) >= 2 and note:
            notes.append(f"{note}（{' / '.join(sorted(set(brands)))}）")
            flagged |= set(brands)
    row["cross_customer_note"] = "；".join(notes)[:MAX_SUMMARY_CHARS * 2] or None
    row["cross_customer_flag"] = bool(notes)
    st["signals"] = len(notes)
    if row["shareable_failed"]:
        st["shareable_failed"] = 1
    return row, flagged, st


In [ ]:
# [c06] pure_spark
# 純 DataFrame 函式：本期相關條目 → 會議端格（一格一列，條目收成 array<struct>）；填報端格；full outer 成 grid。
# 格子數每期百級、條目數百級，grid collect 到 driver 組 prompt 是刻意的（LLM 呼叫是 I/O 等待）。不 cache（serverless）。
def select_relevant(df_q: DataFrame, brief_month: str) -> DataFrame:
    """v_meeting_items_sb_q（已是 active）→ 本期 sb_relevant 的條目；brand NULL → UNKNOWN（D36）。"""
    return (df_q.where((F.col("brief_month") == F.lit(brief_month)) & F.col("sb_relevant"))
            .withColumn("brand", F.coalesce(F.col("brand"), F.lit(BRAND_UNKNOWN)))
            .select(*ITEM_COLS))


def meeting_cells(df_items: DataFrame) -> DataFrame:
    """→ (app_name, brand, facet, brief_month, window_start, window_end, items, n_items, topic_tags_agg, sensitivity_max)。
    items = array<struct<CELL_ITEM_FIELDS>>（順序在 driver 由 pick_items 決定）。"""
    sens_num = F.coalesce(F.element_at(F.create_map(*[F.lit(x) for kv in SENS_ORDER.items() for x in kv]),
                                       F.col("sensitivity")), F.lit(-1))
    back = F.create_map(*[F.lit(x) for k, v in SENS_ORDER.items() for x in (v, k)])
    return (df_items.withColumn("_sens", sens_num)
            .groupBy("app_name", "brand", "facet", "brief_month")
            .agg(F.min("window_start").alias("window_start"), F.max("window_end").alias("window_end"),
                 F.collect_list(F.struct(*CELL_ITEM_FIELDS)).alias("items"),
                 F.count("*").cast("int").alias("n_items"),
                 F.array_sort(F.array_distinct(F.collect_list("topic_tag_primary"))).alias("topic_tags_agg"),
                 F.element_at(back, F.max("_sens")).alias("sensitivity_max")))


def sb_cells(df_brief: DataFrame, brief_month: str) -> DataFrame:
    """填報端有內容的格（含 ALL）。"""
    return (df_brief.where((F.col("brief_month") == F.lit(brief_month)) & F.col("sb_text").isNotNull())
            .select("app_name", "brand", "facet", "brief_month", "sb_text", "sb_source"))


def build_grid(df_m: DataFrame, df_sb: DataFrame) -> DataFrame:
    """會議端 ⋈ 填報端（full outer on app × brand × facet × brief_month）。relation_rule：單邊 → meeting_only / sb_only；
    兩邊 → NULL（交 LLM）。窗口：填報端獨有的格補同 app 的窗口（同期任一會議格的）。"""
    keys = ["app_name", "brand", "facet", "brief_month"]
    j = df_m.join(df_sb, keys, "full_outer")
    has_m, has_sb = F.col("n_items").isNotNull(), F.col("sb_text").isNotNull()
    rel = (F.when(has_m & ~has_sb, F.lit("meeting_only"))
           .when(~has_m & has_sb, F.lit("sb_only")))
    w = Window.partitionBy("app_name", "brief_month")
    return (j.withColumn("relation_rule", rel)
            .withColumn("has_meeting", has_m).withColumn("has_sb", has_sb)
            .withColumn("window_start", F.coalesce(F.col("window_start"), F.min("window_start").over(w)))
            .withColumn("window_end", F.coalesce(F.col("window_end"), F.max("window_end").over(w)))
            .withColumn("n_items", F.coalesce(F.col("n_items"), F.lit(0)))
            .withColumn("sb_source", F.coalesce(F.col("sb_source"), F.lit("none"))))


def grid_to_dicts(rows) -> list[dict]:
    """collect 後的 Row → dict（items 的 struct 也轉 dict）。"""
    out = []
    for r in rows:
        d = r.asDict(recursive=True)
        d["items"] = [dict(it) for it in (d.get("items") or [])]
        out.append(d)
    return out


In [ ]:
# [c07] llm_client
# LLM 接線：Azure Foundry 的 Anthropic Messages 端點，與 task 3 / Stage A 同一組 secret / URL。用 requests（DBR 內建），不 %pip install。
# thinking 關閉、不送 temperature、system 開 prompt caching（system prompt 固定，整期共用）。
# 重試：HTTP 408 / 429 / 5xx / 連線錯誤指數退避 3 次；4xx 其他直接回 error 不重試。JSON / schema 失敗的重試在 [c11]。
import requests

FOUNDRY_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry")
T4_MAX_TOKENS = 4000      # 一格兩段摘要 ＋ 理由，或摘要模式 ≤ 12 格；截斷視為失敗（[c11]）
HTTP_TIMEOUT_S = 180


def call_llm(messages: list[dict], system_prompt: str, model: str, cache: bool) -> dict:
    """回傳 {"text", "stop_reason", "usage": {input, output, cache_read, cache_write}, "latency_s", "error"}；
    error 非 None 代表端點層失敗（已重試），text 為 None。"""
    system = [{"type": "text", "text": system_prompt}]
    if cache:
        system[0]["cache_control"] = {"type": "ephemeral"}
    body = {"model": model, "max_tokens": T4_MAX_TOKENS, "system": system,
            "messages": messages, "thinking": {"type": "disabled"}}
    headers = {"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"}
    t0 = time.time()
    err = None
    for attempt in range(3):
        try:
            resp = requests.post(FOUNDRY_URL, headers=headers, json=body, timeout=HTTP_TIMEOUT_S)
            if resp.status_code in (408, 429) or resp.status_code >= 500:
                err = f"HTTP {resp.status_code}: {resp.text[:200]}"
            elif resp.status_code >= 400:
                return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2),
                        "error": f"HTTP {resp.status_code}: {resp.text[:300]}"}
            else:
                data = resp.json()
                text = "\n".join(b.get("text", "") for b in data.get("content", []) if b.get("type") == "text")
                u = data.get("usage") or {}
                usage = {"input": u.get("input_tokens", 0) or 0, "output": u.get("output_tokens", 0) or 0,
                         "cache_read": u.get("cache_read_input_tokens", 0) or 0,
                         "cache_write": u.get("cache_creation_input_tokens", 0) or 0}
                return {"text": text, "stop_reason": data.get("stop_reason"), "usage": usage,
                        "latency_s": round(time.time() - t0, 2), "error": None}
        except (requests.RequestException, ValueError) as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
        time.sleep(5 * (2 ** attempt))
    return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2), "error": err}


print("llm client ready:", FOUNDRY_URL.split("/anthropic")[0], "| model:", cfg["model"], "| cache:", cfg["prompt_cache"])


In [ ]:
# [c10] load_source
# 讀 T3 標註（本期、相關）與填報文字 → 會議端格 / 填報端格 → grid collect 到 driver → 分三類：兩邊都有（格子模式）、只有會議（摘要模式）、
# 只有填報（規則）；ALL 格另記。每格先挑送 LLM 的條目（pick_items）。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_q, fq_brief, fq_period = fq("v_meeting_items_sb_q"), fq("v_meeting_sb_brief_text"), fq("v_meeting_sb_period")
fq_rec = fq(f"g_meeting_sb_reconcile_{cfg['unit']}")
for t in (fq_q, fq_brief, fq_period, fq_rec):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 004 與 T3"
assert PROMPT_VERSION_IN_CELL == cfg["prompt_version"], (
    f"[c04] 版本 {PROMPT_VERSION_IN_CELL} ≠ widget prompt_version {cfg['prompt_version']}")

bm = cfg["brief_month"]
df_items = select_relevant(spark.table(fq_q), bm)
df_grid = build_grid(meeting_cells(df_items), sb_cells(spark.table(fq_brief), bm))
grid = grid_to_dicts(df_grid.collect())
for g in grid:
    g["items_sent"] = pick_items(g["items"], g["facet"], cfg["max_items_per_cell"], cfg["max_items_per_tag_other"]) if g["items"] else []

cells_both = sorted([g for g in grid if g["relation_rule"] is None and g["brand"] != BRAND_ALL],
                    key=lambda g: (-g["n_items"], g["app_name"], g["brand"], g["facet"]))
cells_meeting_only = sorted([g for g in grid if g["relation_rule"] == "meeting_only"],
                            key=lambda g: (-g["n_items"], g["app_name"], g["brand"], g["facet"]))
cells_sb_only = [g for g in grid if g["relation_rule"] == "sb_only" and g["brand"] != BRAND_ALL]
all_cells = {(g["app_name"], g["facet"]): g for g in grid if g["brand"] == BRAND_ALL}
n_items_total = sum(g["n_items"] for g in grid)
n_items_sent_total = sum(len(g["items_sent"]) for g in grid)
scope = (f"brief_month {bm} | grid {len(grid)} = both {len(cells_both)} + meeting_only {len(cells_meeting_only)}"
         f" + sb_only {len(cells_sb_only)} + ALL(sb) {len(all_cells)} | items {n_items_total}（送 LLM {n_items_sent_total}）")
print(scope)
if not any(g["has_sb"] for g in grid):
    print(f"WARN 本期沒有任何填報文字（brief 還沒填？）：所有格都是 meeting_only，只產摘要、不判 relation；brief 填完後重跑")
n_unknown = sum(g["n_items"] for g in grid if g["brand"] == BRAND_UNKNOWN)
if n_unknown:
    print(f"INFO brand = {BRAND_UNKNOWN}（客戶 NULL 但 app 已解）的相關條目 {n_unknown} 條，當 meeting_only 格（D36）")


In [ ]:
# [c11] run_reconcile
# 三種 LLM 呼叫在 driver 跑（ThreadPool）：(1) 格子模式：兩邊都有的格，一格一次；(2) 摘要模式：只有會議的格，summary_batch 格一次；
# (3) app 層模式：每 app × facet 一次（輸入 (1)(2) 產出的品牌格摘要 ＋ ALL 格填報）。max_cells 是總呼叫數閘門，依 (1)(2)(3) 順序分配。
# JSON / schema 失敗把上一輪輸出當 assistant turn、帶錯誤訊息再試 1 次；輸出被截斷視為失敗。失敗的格不產列（下次再判）。
usage_tot = {"input": 0, "output": 0, "cache_read": 0, "cache_write": 0, "calls": 0, "retries": 0}
latencies: list[float] = []
tot: dict = {}


def _account(r: dict) -> None:
    usage_tot["calls"] += 1
    for k in ("input", "output", "cache_read", "cache_write"):
        usage_tot[k] += (r.get("usage") or {}).get(k, 0)
    latencies.append(r.get("latency_s") or 0.0)


def ask(user_prompt: str) -> tuple[dict | None, str | None]:
    """一次呼叫（含 JSON 重試一次）→ (obj, error)。"""
    messages = [{"role": "user", "content": user_prompt}]
    r1 = call_llm(messages, SYSTEM_PROMPT_T4, cfg["model"], cfg["prompt_cache"])
    _account(r1)
    if r1["error"]:
        return None, r1["error"]
    if r1["stop_reason"] == "max_tokens":
        return None, "output truncated at max_tokens"
    obj, perr = parse_llm_json(r1["text"])
    if obj is not None:
        return obj, None
    usage_tot["retries"] += 1
    messages += [{"role": "assistant", "content": (r1["text"] or "").strip() or "(empty)"},
                 {"role": "user", "content": retry_user_prompt(perr)}]
    r2 = call_llm(messages, SYSTEM_PROMPT_T4, cfg["model"], cfg["prompt_cache"])
    _account(r2)
    if r2["error"]:
        return None, r2["error"]
    if r2["stop_reason"] == "max_tokens":
        return None, "truncated (retry)"
    return parse_llm_json(r2["text"])


def run_parallel(fn, jobs: list) -> list:
    out = [None] * len(jobs)
    if not jobs:
        return out
    with ThreadPoolExecutor(max_workers=cfg["concurrency"]) as ex:
        futs = {ex.submit(fn, j): i for i, j in enumerate(jobs)}
        for fut in as_completed(futs):
            out[futs[fut]] = fut.result()
    return out


budget = cfg["max_cells"] if cfg["max_cells"] > 0 else 10 ** 9
t_start = time.time()
rows: list[dict] = []

# (1) 格子模式
todo_both = cells_both[:budget]
budget -= len(todo_both)


def process_cell(cell: dict):
    obj, err = ask(build_cell_prompt(cell, cell["items_sent"]))
    return postprocess_cell(obj, err, cell, cell["items_sent"], cfg["prompt_version"], cfg["model"])


for row, st in run_parallel(process_cell, todo_both):
    merge_stats(tot, st)
    if row:
        rows.append(row)
tot["cell_skipped_budget"] = len(cells_both) - len(todo_both)
print(f"cell mode: sent {len(todo_both)} / {len(cells_both)} | ok {sum(1 for r in rows)} | {time.time() - t_start:.0f}s", flush=True)

# (2) 摘要模式（只有會議的格；含 UNKNOWN 格）
bs = cfg["summary_batch"]
batches = [[(c, c["items_sent"]) for c in cells_meeting_only[i:i + bs]] for i in range(0, len(cells_meeting_only), bs)]
todo_sum = batches[:budget]
budget -= len(todo_sum)


def process_summary(batch):
    obj, err = ask(build_summary_prompt(batch))
    return postprocess_summary(obj, err, batch)


summaries: dict[str, tuple] = {}
for out, st in run_parallel(process_summary, todo_sum):
    merge_stats(tot, st)
    summaries.update(out)
for cell in cells_meeting_only:
    row = rule_row(cell, "meeting_only", cfg["prompt_version"])
    k = cell_key(cell["app_name"], cell["brand"], cell["facet"])
    if k in summaries:
        row["meeting_summary"], row["meeting_summary_shareable"], row["shareable_failed"] = summaries[k]
        row["model"], row["n_items_sent"] = cfg["model"], len(cell["items_sent"])
    rows.append(row)
tot["summary_batches_skipped_budget"] = len(batches) - len(todo_sum)
print(f"summary mode: batches {len(todo_sum)} / {len(batches)} | cells filled {len(summaries)} / {len(cells_meeting_only)} | {time.time() - t_start:.0f}s", flush=True)

# (3) 只有填報的品牌格：純規則
for cell in cells_sb_only:
    rows.append(rule_row(cell, "sb_only", cfg["prompt_version"]))

# (4) app 層：每 app × facet，有品牌格摘要才跑；ALL 格填報空就 relation = meeting_only（rule）
app_jobs = []
if cfg["app_level"]:
    by_af: dict[tuple, list[dict]] = {}
    for r in rows:
        if r["brand"] != BRAND_ALL and r["meeting_summary"]:
            cell = next(g for g in grid if (g["app_name"], g["brand"], g["facet"]) == (r["app_name"], r["brand"], r["facet"]))
            by_af.setdefault((r["app_name"], r["facet"]), []).append(
                {"brand": r["brand"], "meeting_summary": r["meeting_summary"], "item_ids": (r["meeting_item_ids"] or [])[:5],
                 "item_keys": r["meeting_item_keys"], "topic_tags_agg": r["topic_tags_agg"],
                 "sensitivity_max": r["sensitivity_max"], "items": cell["items_sent"]})
    for (app, facet), bcs in sorted(by_af.items()):
        all_cell = all_cells.get((app, facet)) or {"app_name": app, "brand": BRAND_ALL, "facet": facet, "brief_month": bm,
                                                    "window_start": None, "window_end": None, "sb_text": None,
                                                    "sb_source": "none", "items": [], "topic_tags_agg": None, "sensitivity_max": None}
        if all_cell.get("window_start") is None:  # 合成的 ALL 格補同 app 窗口
            src = next((g for g in grid if g["app_name"] == app and g.get("window_start")), None)
            if src:
                all_cell["window_start"], all_cell["window_end"] = src["window_start"], src["window_end"]
        app_jobs.append((all_cell, bcs))
    app_jobs_todo = app_jobs[:budget]
    budget -= len(app_jobs_todo)

    def process_app(job):
        all_cell, bcs = job
        obj, err = ask(build_app_prompt(all_cell["app_name"], all_cell["facet"], bm, bcs, all_cell.get("sb_text")))
        return postprocess_app(obj, err, all_cell, bcs, cfg["prompt_version"], cfg["model"])

    done_af = set()
    for (all_cell, bcs), (row, flagged, st) in zip(app_jobs_todo, run_parallel(process_app, app_jobs_todo)):
        merge_stats(tot, st)
        if row:
            rows.append(row)
            done_af.add((all_cell["app_name"], all_cell["facet"]))
            for r in rows:
                if r["app_name"] == all_cell["app_name"] and r["facet"] == all_cell["facet"] and r["brand"] in flagged:
                    r["cross_customer_flag"] = True
    tot["app_skipped_budget"] = len(app_jobs) - len(app_jobs_todo)
    # 只有填報的 ALL 格（沒有任何品牌格摘要）→ sb_only
    for (app, facet), g in all_cells.items():
        if (app, facet) not in by_af and (app, facet) not in done_af:
            rows.append(rule_row(g, "sb_only", cfg["prompt_version"]))
    print(f"app mode: sent {len(app_jobs_todo)} / {len(app_jobs)} | {time.time() - t_start:.0f}s", flush=True)
else:
    for g in all_cells.values():
        rows.append(rule_row(g, "sb_only", cfg["prompt_version"]))

df_batch = spark.createDataFrame([tuple(r.get(f) for f in ROW_SCHEMA.fieldNames()) for r in rows], ROW_SCHEMA)
df_batch.createOrReplaceTempView("_w2_t4_batch")
n_batch = len(rows)
batch_keys = [(r["app_name"], r["brand"], r["facet"], r["brief_month"]) for r in rows]
assert len(set(batch_keys)) == n_batch, "本批格子鍵重複（[c10] grid 或 [c11] 組列有 bug）"
print(f"batch rows {n_batch} | calls {usage_tot['calls']} | retries {usage_tot['retries']} | {time.time() - t_start:.0f}s")


In [ ]:
# [c12] gates
# 關卡：端點全掛 → 不寫半批；LLM 格失敗 > 50% 停。只印：relation 分布、shareable 驗證失敗數、摘要缺的格、跨客戶訊號數、
# 預算跳過的格、token 用量與費用；dry_run 再印例外清單預覽（§4.6 順序：conflict → supplement → meeting_only → ALL → corroborate）。
def pct(n, d) -> float | None:
    return round(n / d, 3) if d else None


n_llm_sent = tot.get("cell_sent", 0) + tot.get("summary_cells_sent", 0) + tot.get("app_sent", 0)
n_llm_failed = tot.get("cell_failed", 0) + tot.get("summary_batch_failed", 0) + tot.get("app_failed", 0)
cost = (usage_tot["input"] * PRICE_PER_MTOK["input"] + usage_tot["output"] * PRICE_PER_MTOK["output"]
        + usage_tot["cache_read"] * PRICE_PER_MTOK["cache_read"] + usage_tot["cache_write"] * PRICE_PER_MTOK["cache_write"]) / 1e6
log = {
    "scope": scope, "prompt_version": cfg["prompt_version"], "model": cfg["model"], "batch_rows": n_batch,
    "relation_dist": dict(Counter(r["relation"] for r in rows)),
    "relation_basis_dist": dict(Counter(r["relation_basis"] for r in rows)),
    "cell_mode": {"sent": tot.get("cell_sent", 0), "failed": tot.get("cell_failed", 0), "bad_relation": tot.get("bad_relation", 0),
                  "skipped_budget": tot.get("cell_skipped_budget", 0)},
    "summary_mode": {"cells_sent": tot.get("summary_cells_sent", 0), "filled": tot.get("summary_filled", 0),
                     "missing": tot.get("summary_missing", 0), "batches_failed": tot.get("summary_batch_failed", 0),
                     "batches_skipped_budget": tot.get("summary_batches_skipped_budget", 0)},
    "app_mode": {"sent": tot.get("app_sent", 0), "failed": tot.get("app_failed", 0), "signals": tot.get("signals", 0),
                 "skipped_budget": tot.get("app_skipped_budget", 0)},
    "shareable_failed": tot.get("shareable_failed", 0),
    "rows_without_summary": sum(1 for r in rows if r["relation"] != "sb_only" and not r["meeting_summary"]),
    "cross_customer_flagged": sum(1 for r in rows if r["cross_customer_flag"] and r["brand"] != BRAND_ALL),
    "sensitivity_max_dist": dict(Counter(r["sensitivity_max"] for r in rows)),
    "usage": {**usage_tot, "latency_p50_s": (sorted(latencies)[len(latencies) // 2] if latencies else None),
              "latency_max_s": (max(latencies) if latencies else None), "est_cost_usd": round(cost, 3)},
}
print(json.dumps(log, ensure_ascii=False, default=str, indent=1))
if tot.get("errors"):
    print("失敗（最多 5）：", tot["errors"][:5])
if n_llm_sent:
    assert n_llm_failed < max(1, usage_tot["calls"]), "LLM 端點整體不可用：全部呼叫失敗，不寫半批"
    assert pct(n_llm_failed, max(1, usage_tot["calls"])) <= 0.5, "LLM 失敗超過一半：先查端點 / prompt 再跑（失敗的格下次會再判）"
if log["shareable_failed"]:
    print(f"WARN shareable 驗證失敗 {log['shareable_failed']} 格：外送版含 S2 數字，已置 NULL；看 prompt 的 S2 規則")
if n_batch == 0:
    print("本期 0 格：T3 還沒標本期、或本期沒有相關條目也沒有填報。後面 cell 不寫表。")
if cfg["dry_run"] and n_batch:
    rank = {"conflict": 1, "supplement": 2, "meeting_only": 3, "corroborate": 4, "sb_only": 5}
    print("\ndry_run = true：不寫表。例外清單預覽（已 token 化，無人名）：")
    (df_batch.withColumn("_rank", F.when(F.col("brand") == BRAND_ALL, 3.5).otherwise(
        F.coalesce(*[F.when(F.col("relation") == k, F.lit(v)) for k, v in rank.items()], F.lit(9))))
     .where(F.col("relation") != "sb_only")
     .select("app_name", "brand", "facet", "relation", "relation_basis", "n_items", "n_items_sent", "sensitivity_max",
             "shareable_failed", "cross_customer_flag", F.substring("sb_text", 1, 80).alias("sb_head"),
             F.substring("meeting_summary", 1, 160).alias("summary_head"),
             F.substring("meeting_summary_shareable", 1, 120).alias("shareable_head"),
             F.substring("rationale", 1, 100).alias("rationale_head"), F.substring("cross_customer_note", 1, 120).alias("signal_head"))
     .orderBy("_rank", "app_name", F.desc("n_items")).show(40, truncate=160, vertical=True))


In [ ]:
# [c20] write_reconcile
# MERGE on (app_name, brand, facet, brief_month)：UPDATE / INSERT 都只碰 WRITE_COLS（PROTECTED_COLS = PM 審核欄不在清單裡，既有值原封不動；
# 新列五欄 NULL）；created_at 只在 INSERT 填，updated_at 每次填。
n_review_before = None
if not cfg["dry_run"] and n_batch > 0:
    n_review_before = spark.table(fq_rec).where(F.col("review_verdict").isNotNull() | F.col("review_note").isNotNull()).count()
    data_cols = [c for c in WRITE_COLS if c not in ("created_at", "updated_at")]
    key_cols = ("app_name", "brand", "facet", "brief_month")
    upd = ", ".join(f"t.{c} = s.{c}" for c in data_cols if c not in key_cols)
    spark.sql(f"""
        MERGE INTO {fq_rec} t
        USING _w2_t4_batch s
        ON t.app_name = s.app_name AND t.brand = s.brand AND t.facet = s.facet AND t.brief_month = s.brief_month
        WHEN MATCHED THEN UPDATE SET {upd}, t.updated_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT ({", ".join(data_cols)}, created_at, updated_at)
            VALUES ({", ".join(f"s.{c}" for c in data_cols)}, current_timestamp(), current_timestamp())
    """)
    print(f"merged {n_batch} rows →", fq_rec, f"| reviewed rows before: {n_review_before}")
else:
    print("skip write（dry_run 或本期 0 格）")


In [ ]:
# [c30] check_target
# 寫入後驗證（用 [c11] 釘住的 batch_keys）：本批格都在表且版本對；鍵無重複；PM 審核列數沒變少；relation 合法、llm 判的格都有摘要；
# 本期表裡有、本批沒有的格（上次有、這次沒有：品牌或條目消失）只印數量，不刪（PM 可能審過）。最後印本期 relation 分布。
if not cfg["dry_run"] and n_batch > 0:
    rec = spark.table(fq_rec)
    keys_df = spark.createDataFrame(batch_keys, "app_name string, brand string, facet string, brief_month string")
    got = rec.join(keys_df, ["app_name", "brand", "facet", "brief_month"], "left_semi")
    month = rec.where(F.col("brief_month") == F.lit(cfg["brief_month"]))
    chk = {
        "batch_rows_in_table": got.count(), "expected": n_batch,
        "batch_version_ok": got.where(F.col("prompt_version") == F.lit(cfg["prompt_version"])).count(),
        "dup_key": rec.groupBy("app_name", "brand", "facet", "brief_month").count().where("count > 1").count(),
        "review_before": n_review_before,
        "review_after": rec.where(F.col("review_verdict").isNotNull() | F.col("review_note").isNotNull()).count(),
        "bad_relation": got.where(~F.col("relation").isin(*RELATIONS)).count(),
        "llm_without_summary": got.where((F.col("relation_basis") == "llm") & F.col("meeting_summary").isNull()).count(),
        "stale_cells_in_month": month.join(keys_df, ["app_name", "brand", "facet", "brief_month"], "left_anti").count(),
        "table_total": rec.count(),
        "month_relation_dist": {r["relation"]: r["count"] for r in month.groupBy("relation").count().collect()},
    }
    print(chk)
    assert chk["batch_rows_in_table"] == chk["expected"] == chk["batch_version_ok"], "本批格沒全寫進表或版本不對"
    assert chk["dup_key"] == 0, "格子鍵重複（MERGE 鍵壞了）"
    assert chk["review_after"] >= chk["review_before"], "PM 審核列數變少：MERGE 碰到審核欄"
    assert chk["bad_relation"] == 0 and chk["llm_without_summary"] == 0, "relation 不合法或 llm 格沒摘要"
    if chk["stale_cells_in_month"]:
        print(f"INFO 本期有 {chk['stale_cells_in_month']} 格上次有、這次沒產（不刪；要清就手動 DELETE 未審的）")
